# Perkenalan

In [74]:
'''
=================================================
Live Code 3

Nama  : Chandra Julian Adhiyasa
Batch : CODA-RMT-016

Program ini dibuat untuk memenuhi projek milestone dengan melakukan ETL meliputi web scraping, data transformation, dan load to postgresql
=================================================
'''

'\n=================================================\nLive Code 3\n\nNama  : Chandra Julian Adhiyasa\nBatch : CODA-RMT-016\n\nProgram ini dibuat untuk memenuhi projek milestone dengan melakukan ETL meliputi web scraping, data transformation, dan load to postgresql\n=================================================\n'

# Answer

## Extract - Web Scraping

In [ ]:
#Import library
import pandas as pd
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Proses Web Scraping

# Inisiasi variabel yang akan menyimpan value hasil dari scraping web eneba
name = []
ori_price = []
disc_price = []
disc = []
region = []
platforms = []
review = []
img = []
link = []

# inisiasi variabel driver yang berisi method webdriver yang akan menjalankan browser google chrome 
driver = webdriver.Chrome()

# dikarenakan webnya bersifat pagination, maka dilakukan perulangan untuk melakukan load data dan load halaman sebanyak 10x
for page in range(1,10) :
    ''' 
    menyimpan link web eneba ke dalam variabel url dengan memanfaatkan 'f' atau string formatted untuk dapat memasukkan variabel page
    ke dalam link agar halaman web berubah ke halaman berikutnya
    '''
    url = f'https://www.eneba.com/id-id/store/all?page={page}&types[]=game'

    # Menjalankan url web eneba di browser google chrome dengan jeda load selama 7 detik
    driver.get(url)
    time.sleep(7)

    '''
    webdriver akan mengambil isi html dari web tokopedia yang disimpan ke dalam variabel html.
    lalu beautifulsoup akan menguraikan html agar elemen di dalamnya dapat terbaca
    '''
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # dengan menggunakan beautifulsoup, akan dicari elemen yang memiliki tag 'div' dengan dengan attribut 'class' yang memiliki value 'uy1qit'
    items = soup.find_all('div', {'class' : 'uy1qit'})

    '''
    Perulangan di bawah dilakukan untuk mendapatkan setiap data sesuai dengan elemen yang diinput.
    data-data tersebut akan disimpan ke dalam setiap variabel agar memudahkan untuk mengaksesnya kembali
    '''
    for item in items :
        el_name = item.find('span', {'class' : 'YLosEL'})
        el_ori_price = item.find('span', {'class' : 'L5ErLT bmxuMu'})
        el_disc_price = item.find('span', {'class' : 'DTv7Ag'})
        el_disc = item.find('span', {'class' : 'PIG8fA'})
        el_region = item.find('div', {'class' : 'Pm6lW1'})
        el_platforms = item.find('span', {'class' : 'uAsjsO'})
        el_review = item.find('span', {'class' : 'BwtiXe'})
        el_img = item.find('img', {'class' : 'LBwiWP v5wuNi'})
        el_link = item.find('a', {'class' : 'GZjXOw'})

        '''
        method try digunakan dengan tujuan untuk memasukkan data-data yang telah didapatkan dari masing-masing elemen ke dalam
        variabel kosong yang sudah diinisiasi di awal.
        Jika terdpat nilai null, maka method except akan digunakan untuk memasukkan nilai None atau Nan
        '''
        try :
            name.append(el_name.get_text())
        except :
            name.append(None)
        try :
            ori = el_ori_price.get_text()
        except :
            ori = None
        try :
            discp = el_disc_price.get_text()
        except :
            discp = None

        '''
        Proses kondisi di bawah ini dilakukan karena terdapat beberapa product yang tidak memiliki diskon
        sehingga awalnya data pada elemen ori_price kosong sementara terdapat data harga pada elemen disc_price.add()
        Dengan menggunakan bantuan if else, dapat dilakukan proses jika maka dimana :
            1. Jika pada sebuah item terdapat harga original dan harga diskon, maka input harga original ke elemen ori_price
                dan harga diskon ke elemen disc_price
            2. Jika hanya terdapat nilai pada bagian tag untuk harga diskon, sementara tidak ada value pada bagian tag harga original,
                maka masukkan elemen pada tag harga diskon ke dalam elemen ori_price
            3. Jika keduanya tidak ada, maka ganti valuenya menjadi none atau Nan
        '''
        if ori and discp :
            ori_price.append(ori)
            disc_price.append(discp)
        elif discp and not ori :
            ori_price.append(discp)
            disc_price.append(None)
        else :
            ori_price.append(None)
            disc_price.append(None)

        try :
            disc.append(el_disc.get_text())
        except :
            disc.append(None)
        try :
            region.append(el_region.get_text())
        except :
            region.append(None)
        try :
            platforms.append(el_platforms.get_text())
        except :
            platforms.append(None)
        try :
            review.append(el_review.get_text())
        except :
            review.append(None)
        try :
            img.append(el_img['src'])
        except :
            img.append(None)
        try :
            link.append(el_link['href'])
        except :
            link.append(None)

# Mengakhiri proses web scraping
driver.quit()

# Membuat sebuah dataframe yang disimpan dalam variabel df dengan isi dari dataframenya adalah data-data yang telah didapat dari proses scraping
df = pd.DataFrame({
    'name' : name,
    'ori_price' : ori_price,
    'disc_price' : disc_price,
    'disc' : disc,
    'region' : region,
    'platforms' : platforms,
    'review' : review,
    'img' : img,
    'link' : link,
})

# Export hasil pembuatan dataframe ke dalam file csv
df.to_csv('hasil_scraping_eneba.csv', index = False)

# Menampilkan dataframe
df

,name,ori_price,disc_price,disc,region,platforms,review,img,link
0,Grand Theft Auto V Enhanced (PC) Rockstar Game...,"Rp 188.584,55",None,None,Global,Rockstar Games Launcher,5121,https://imgproxy.eneba.games/rx3qzhF6ncw4ymmSa...,/id-id/rockstar-social-club-grand-theft-auto-v...
1,Minecraft: Java & Bedrock Edition (PC) Officia...,"Rp 594.403,74","Rp 265.714,84",-55%,Global,Windows Store,99607,https://imgproxy.eneba.games/HFctKs2s7yb7zhAs0...,/id-id/other-minecraft-java-bedrock-edition-pc...
2,Grand Theft Auto V: Premium Online Edition Roc...,"Rp 600.990,61","Rp 188.584,55",-69%,Global,Rockstar Games Launcher,140494,https://imgproxy.eneba.games/TXXnT0z8tVRmLfaLr...,/id-id/rockstar-social-club-rockstar-social-cl...
3,Minecraft: Java & Bedrock Edition (PC) Windows...,"Rp 264.531,26",None,None,Global,Windows Store,25031,https://imgproxy.eneba.games/HFctKs2s7yb7zhAs0...,/id-id/windows-store-minecraft-java-bedrock-ed...
4,Red Dead Redemption 2 Rockstar Games Launcher ...,"Rp 1.202.582,61","Rp 236.125,22",-80%,Global,Rockstar Games Launcher,27381,https://imgproxy.eneba.games/5KH5PbydU4nJtjaRR...,/id-id/rockstar-social-club-red-dead-redemptio...
...,...,...,...,...,...,...,...,...,...
67,Batman: Arkham Asylum (GOTY) Steam Key GLOBAL,"Rp 387.483,05","Rp 35.113,02",-91%,Global,Steam,7458,https://imgproxy.eneba.games/X0na7NFGNjxHeodGf...,/id-id/steam-batman-arkham-asylum-goty-steam-k...
68,Forza Horizon 5 PC/XBOX LIVE Key GLOBAL,"Rp 1.356.675,26","Rp 517.029,40",-62%,Global,Xbox Live,9794,https://imgproxy.eneba.games/vgNClViQrBpYQSOme...,/id-id/xbox-forza-horizon-5-pc-xbox-live-key-g...
69,GTA V Premium Online Edition & Great White Sha...,"Rp 349.354,85",None,None,Global,Rockstar Games Launcher,11345,https://imgproxy.eneba.games/8t8CjodUP6tJbxN14...,/id-id/steam-grand-theft-auto-v-premium-online...
70,Batman: Arkham Knight (Premium Edition) Steam ...,"Rp 32.154,06",None,None,Global,Steam,6913,https://imgproxy.eneba.games/zi2uCaiXovBYT58Bb...,/id-id/steam-batman-arkham-knight-premium-edit...


## Data Transformation

In [ ]:
# Muat data csv hasil export dari proses web scraping dengan memanfaatkan library pandas
df = pd.read_csv('hasil_scraping_eneba.csv')
# Tampilkan informasi struktur tabel atau dataframe
df.info()

'''
Dari hasil scraping, didapatkan 72 baris data dengan 9 kolom.
8 kolom memiliki tipe data object dan 1 kolom bertipe integer.
pada kolom disc_price dan disc terdapat beberapa data tanpa value (Nan)

Dengan hasil struktur di atas, dapat disimpulkan bahwa data yang didapat masih lah berupa data kotor.
misalnya terdapat value null di kolom disc_price dan disc, atau kolom ori_price yang seharusnya bertipe integer atau float tetapi malah bertipe object
'''

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        72 non-null     object
 1   ori_price   72 non-null     object
 2   disc_price  53 non-null     object
 3   disc        53 non-null     object
 4   region      72 non-null     object
 5   platforms   72 non-null     object
 6   review      72 non-null     int64 
 7   img         72 non-null     object
 8   link        72 non-null     object
dtypes: int64(1), object(8)
memory usage: 5.2+ KB


'\nDari hasil scraping, didapatkan 72 baris data dengan 9 kolom.\n8 kolom memiliki tipe data object dan 1 kolom bertipe integer.\npada kolom disc_price dan disc terdapat beberapa data tanpa value (Nan)\n\nDengan hasil struktur di atas, dapat disimpulkan bahwa data yang didapat masih lah berupa data kotor.\nmisalnya terdapat value null di kolom disc_price dan disc, atau kolom ori_price yang seharusnya bertipe integer atau float tetapi malah bertipe object\n'

In [4]:
'''
Proses-proses di bawah ini adalah proses transformasi untuk mengubah value string yang tidak diperlukan guna mengubah
tipe data object pada kolom-kolom di bawah menjadi tipe data angka seperti float.
Method 'str.replace' digunakan untuk mengubah atau bahkan menghapus string yang tidak diperlukan.
Method fillna() digunakan untuk mengisi value null dengan nilai 0
Method astype digunakan untuk mengubah tipe data pada suatu kolom
'''

df['ori_price'] = (
    df['ori_price'].str.replace('Rp\xa0', '').str.replace('.', '').str.replace(',', '.').fillna(0).astype(float)
)

df['disc_price'] = (
    df['disc_price'].str.replace('Rp\xa0', '').str.replace('.', '').str.replace(',', '.').fillna(0).astype(float)
)

df['disc'] = (
    df['disc'].str.replace('-', '').str.replace('%', '').fillna(0).astype(float) / 100
)

In [ ]:
''' 
Melihat informasi struktur dataframe setelah dimanipulasi dengan pandas.
Dapat dilihat jika sudah tidak ada lagi value null pad akolom disc_price dan disc.
tipe data kolom ori_price, disc_price, dan disc telah berganti dari object menjadi float
'''
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        72 non-null     object 
 1   ori_price   72 non-null     float64
 2   disc_price  72 non-null     float64
 3   disc        72 non-null     float64
 4   region      72 non-null     object 
 5   platforms   72 non-null     object 
 6   review      72 non-null     int64  
 7   img         72 non-null     object 
 8   link        72 non-null     object 
dtypes: float64(3), int64(1), object(5)
memory usage: 5.2+ KB


In [ ]:
# Export hasil transformasi data dengan pandas menjadi file csv sebagai data bersih
df.to_csv('hasil_akhir.csv', index=False)